# Lab 09-01: Deploy an Agent to Cloud Run and Release a Canary

**Book chapter:** Chapter 09: Deploying, Operating, and Observing Agents  
**Estimated time:** 35 minutes

## Overview

Cymbal Retail's platform team wants to run the store concierge on Cloud Run and release
every new version safely. In this lab, you package an ADK agent as a container, deploy it
to Cloud Run with its own service account, and call it over HTTPS. You then deploy a
second version as a canary with no traffic, test it, send it 10 percent of the traffic,
promote it, and roll it back.

## Learning Objectives

By the end of this lab, you will be able to:

- Package an ADK agent as a web service with a Dockerfile
- Deploy to Cloud Run from source with a dedicated service account
- Call a private Cloud Run service with an identity token
- Release a canary with a tag URL and traffic splitting
- Promote and roll back a release with traffic changes

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder
- The **Owner** role on the lab project, because the lab creates a service account and grants it a role
- The Google Cloud CLI, which Colab and Cloud Shell already provide

**Cost:** A few cents for two container builds. The service scales to zero when idle. Run the **Clean Up** cell at the end.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Term | Meaning |
|---|---|
| **Revision** | An immutable snapshot of the service's code and settings, created by each deployment |
| **Traffic split** | The percentage of requests that each revision receives |
| **Tag** | A name for a revision that gives it its own URL, even with no traffic |
| **Canary** | A new revision that receives a small share of traffic while it is watched |
| **Rollback** | Sending all traffic back to the previous revision |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk httpx

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Write the agent service files

The service has three parts, which the cell writes to the `concierge_service` folder:

| File | Purpose |
|---|---|
| `concierge/agent.py` | The ADK agent. `TOOLS` is replaced with the tools of each version. |
| `main.py` | Serves the agent with ADK's FastAPI app (`/run`) and adds `/release`, which reports the version |
| `Dockerfile` | Builds the container image that Cloud Run runs |

Version 1 has only `get_store_hours`. The sessions are kept in memory, which is fine for a
lab; in production, Cloud Run instances are replaced at any time, so sessions belong in a
database or in Agent Platform's managed sessions.

In [ ]:
import collections
import os
import pathlib
import shutil
import subprocess
import time
import uuid

import httpx

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
REGION = "us-central1"
SERVICE = "cymbal-concierge"
APP = pathlib.Path("concierge_service")

AGENT = '''import os
from google.adk.agents import Agent

HOURS = {"seattle": "9 AM to 9 PM", "portland": "10 AM to 8 PM"}
STOCK = {"trailhead boots": 12, "rain jacket": 0}


def get_store_hours(city: str) -> dict:
    """Returns the opening hours of the Cymbal Retail store in a city."""
    return {"city": city, "hours": HOURS.get(city.lower(), "unknown")}


def check_stock(item: str) -> dict:
    """Returns how many units of an item are in stock online."""
    return {"item": item, "units": STOCK.get(item.lower(), 0)}


root_agent = Agent(
    name="concierge", model=os.environ["GEMINI_MODEL"],
    instruction="You are the Cymbal Retail concierge. Use your tools, "
                "and answer in one sentence. If you have no tool for a "
                "question, say that you can't help with it yet.",
    tools=[TOOLS])
'''

MAIN = '''import os
from google.adk.cli.fast_api import get_fast_api_app

app = get_fast_api_app(
    agents_dir=os.path.dirname(os.path.abspath(__file__)),
    session_service_uri="memory://",  # Use a database in production.
    auto_create_session=True, web=False)


@app.get("/release")  # ADK already uses /version for its own version.
def release():
    return {"version": os.environ.get("AGENT_VERSION"),
            "revision": os.environ.get("K_REVISION", "local")}
'''

DOCKERFILE = '''FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD exec uvicorn main:app --host 0.0.0.0 --port $PORT
'''


def write_app(tools):
    """Writes the agent package, the web server, and the Dockerfile."""
    package = APP / "concierge"
    package.mkdir(parents=True, exist_ok=True)
    (package / "__init__.py").write_text("from . import agent\n")
    (package / "agent.py").write_text(AGENT.replace("TOOLS", tools))
    (APP / "main.py").write_text(MAIN)
    (APP / "requirements.txt").write_text("google-adk==2.15.0\n")
    (APP / "Dockerfile").write_text(DOCKERFILE)


write_app("get_store_hours")  # Version 1 has one tool.
print("Wrote", sorted(str(p) for p in APP.rglob("*") if p.is_file()))

## Step 2: Create a runtime identity and deploy version 1

The `gcloud` helper runs Google Cloud CLI commands from Python. The cell creates a
service account that can only call Gemini (`roles/aiplatform.user`) and deploys the
service with it. Important settings:

| Flag | Why |
|---|---|
| `--no-allow-unauthenticated` | Only callers with an identity token can reach the agent |
| `--min-instances=0` | Scales to zero when idle, so the lab costs almost nothing |
| `--concurrency=20` | Requests that one instance handles at a time; agents mostly wait on I/O |
| `--timeout=300` | Long enough for a multi-step agent turn |

> **Note:** The build takes about 3 to 5 minutes. If it fails with a permission error, the
> Compute Engine default service account, which builds source deployments, needs the
> Cloud Run Builder role. Run the **If the Build Fails** cell after this step, and then
> rerun this cell.

In [ ]:
def gcloud(*args):
    """Runs a gcloud command in the lab project and returns its output."""
    result = subprocess.run(
        [shutil.which("gcloud") or "gcloud", *args, f"--project={PROJECT}"],
        capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-1500:])
    return result.stdout.strip()


SA = f"concierge-runtime@{PROJECT}.iam.gserviceaccount.com"
if not gcloud("iam", "service-accounts", "list", f"--filter=email:{SA}",
              "--format=value(email)"):
    gcloud("iam", "service-accounts", "create", "concierge-runtime",
           "--display-name=Cymbal concierge runtime")
for attempt in range(6):  # A new service account can take a minute.
    try:
        gcloud("projects", "add-iam-policy-binding", PROJECT,
               f"--member=serviceAccount:{SA}",
               "--role=roles/aiplatform.user", "--condition=None")
        break
    except RuntimeError:
        time.sleep(10)
print("Runtime service account:", SA)

ENV = ",".join([
    "GOOGLE_GENAI_USE_VERTEXAI=TRUE", f"GOOGLE_CLOUD_PROJECT={PROJECT}",
    "GOOGLE_CLOUD_LOCATION=global", f"GEMINI_MODEL={MODEL}"])


def deploy(version, *flags):
    """Builds the container from source and deploys a new revision."""
    print(f"Deploying {version}. The build takes about 3 to 5 minutes...")
    gcloud("run", "deploy", SERVICE, f"--source={APP}",
           f"--region={REGION}", f"--service-account={SA}",
           "--no-allow-unauthenticated", "--memory=1Gi",
           "--concurrency=20", "--min-instances=0", "--max-instances=2",
           "--timeout=300", f"--set-env-vars={ENV},AGENT_VERSION={version}",
           "--quiet", *flags)
    revision = gcloud("run", "services", "describe", SERVICE,
                      f"--region={REGION}",
                      "--format=value(status.latestCreatedRevisionName)")
    print(f"{version} is revision {revision}")
    return revision


STABLE = deploy("v1")

**Expected result:** `v1 is revision cymbal-concierge-00001-...`.

### If the Build Fails

Run the next cell only if Step 2 failed with a permission error, and then rerun Step 2. It
grants the Cloud Run Builder role to the Compute Engine default service account, which
builds source deployments.

In [ ]:
# Needed only if the build in Step 2 failed with a permission error.
NUMBER = gcloud("projects", "describe", PROJECT,
                "--format=value(projectNumber)")
gcloud("projects", "add-iam-policy-binding", PROJECT,
       f"--member=serviceAccount:{NUMBER}-compute@developer.gserviceaccount.com",
       "--role=roles/run.builder", "--condition=None")
print("The build service account now has the Cloud Run Builder role.")

## Step 3: Call the deployed agent

The `ask` helper posts a question to the agent's `/run` endpoint with your identity token
and prints the tools that the agent called and its answer.

In [ ]:
URL = gcloud("run", "services", "describe", SERVICE, f"--region={REGION}",
             "--format=value(status.url)")
TOKEN = gcloud("auth", "print-identity-token")  # Proves who is calling.
HEADERS = {"Authorization": f"Bearer {TOKEN}"}


def ask(url, question):
    """Sends one question to the agent's /run endpoint."""
    body = {"app_name": "concierge", "user_id": "lab-user",
            "session_id": str(uuid.uuid4()),
            "new_message": {"role": "user", "parts": [{"text": question}]}}
    response = httpx.post(f"{url}/run", json=body, headers=HEADERS,
                          timeout=180)
    events = response.raise_for_status().json()
    parts = [p for e in events if "content" in e
             for p in e["content"].get("parts", [])]
    tools = [p["functionCall"]["name"] for p in parts
             if "functionCall" in p]
    answer = [p["text"] for p in parts if p.get("text")][-1]
    print(f"Q: {question}\n   tools={tools}\n   A: {answer}")


def version(url):
    """Returns the agent version of the revision that served the request."""
    response = httpx.get(f"{url}/release", headers=HEADERS, timeout=180)
    return response.raise_for_status().json()["version"]


print("Service URL:", URL, "| version:", version(URL))
ask(URL, "When does the Seattle store open?")
ask(URL, "Are the Trailhead boots in stock?")

**Expected result:** version `v1`; the store-hours question uses `get_store_hours`, and
the agent says that it can't help with the stock question yet. The first request can take
a few extra seconds while an instance starts.

## Step 4: Deploy version 2 as a canary that receives no traffic

Version 2 adds `check_stock`. It is deployed with `--no-traffic --tag=canary`, so users
still reach version 1, but you can test version 2 through its **tag URL**, which starts with
`canary---`.

In [ ]:
write_app("get_store_hours, check_stock")  # Version 2 adds a tool.
CANARY = deploy("v2", "--no-traffic", "--tag=canary")
CANARY_URL = URL.replace("https://", "https://canary---")
print("Canary URL:", CANARY_URL, "| version:", version(CANARY_URL))
ask(CANARY_URL, "Are the Trailhead boots in stock?")

**Expected result:** the canary reports version `v2` and answers the stock question with `check_stock`.

## Step 5: Send 10 percent of the traffic to the canary

`set_traffic` changes the split between revisions, and `sample` sends requests to the main
URL and counts which version served each one.

In [ ]:
def set_traffic(*targets):
    """Sets the traffic split, for example 'REVISION=90'."""
    gcloud("run", "services", "update-traffic", SERVICE,
           f"--region={REGION}", "--to-revisions=" + ",".join(targets))
    print("Traffic:", ", ".join(targets))


def sample(requests=40):
    """Counts which version serves each of a number of requests."""
    counts = collections.Counter(version(URL) for _ in range(requests))
    print(f"Served by: {dict(sorted(counts.items()))}\n")


set_traffic(f"{STABLE}=90", f"{CANARY}=10")
sample()

**Expected result:** about 36 requests served by `v1` and 4 by `v2`. The exact numbers vary because Cloud Run routes each request independently.

## Step 6: Promote the canary, and then roll it back

When the canary's metrics look healthy, promote it. If they don't, roll back. Both are
traffic changes, so they take seconds and need no new build.

In [ ]:
set_traffic(f"{CANARY}=100")  # Promote: the canary is healthy.
sample(10)
set_traffic(f"{STABLE}=100")  # Roll back: a traffic change, not a build.
sample(10)

**Expected result:** 10 requests served by `v2`, and then 10 by `v1`.

## Step 7: Experiment

Run the next cell to list the revisions and their traffic, send half of the traffic to the
revision tagged `canary`, and sample the split again.

In [ ]:
print(gcloud("run", "revisions", "list", f"--service={SERVICE}",
             f"--region={REGION}"))
gcloud("run", "services", "update-traffic", SERVICE, f"--region={REGION}",
       "--to-tags=canary=50")
sample()

## Check Your Understanding

**1. Why deploy the canary with --no-traffic and a tag?**

<details><summary>Show answer</summary>

So that you can test the new revision in production through its tag URL before any user reaches it, and then shift traffic gradually.

</details>

**2. Why is a rollback on Cloud Run fast?**

<details><summary>Show answer</summary>

Each deployment creates an immutable revision, so rolling back only changes the traffic split; nothing is rebuilt.

</details>

**3. Why should the agent's sessions not stay in memory in production?**

<details><summary>Show answer</summary>

Cloud Run instances are stateless and can be replaced or scaled at any time, so session state must be kept in a database or managed session service.

</details>

## Clean Up

If you plan to do Lab 09-02 next, you can clean up after it instead. Otherwise, run the next
cell to delete the Cloud Run service, its container image, and its service account.

In [ ]:
import os
import shutil
import subprocess

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
REGION = "us-central1"
IMAGE = (f"{REGION}-docker.pkg.dev/{PROJECT}/cloud-run-source-deploy/"
         "cymbal-concierge")
SA = f"concierge-runtime@{PROJECT}.iam.gserviceaccount.com"
for label, command in [
        ("Cloud Run service", ["run", "services", "delete",
                               "cymbal-concierge", f"--region={REGION}"]),
        ("Container image", ["artifacts", "docker", "images", "delete",
                             IMAGE, "--delete-tags"]),
        ("Service account", ["iam", "service-accounts", "delete", SA])]:
    result = subprocess.run(
        [shutil.which("gcloud") or "gcloud", *command, "--quiet",
         f"--project={PROJECT}"], capture_output=True, text=True)
    print(label, "deleted" if result.returncode == 0 else "not found")

## Summary

You deployed an ADK agent to Cloud Run with a least-privilege service account, released a
second version as a canary, and promoted and rolled it back with traffic changes.